# GST invoice OCR: baseline run on Google Drive

**What this does**
1. Mounts your Google Drive.
2. Unzips the invoice sample set from `MyDrive/gst-engine/`.
3. Runs PaddleOCR (Apache-2.0 code and weights) on the invoice photos.
4. Saves the OCR text and a field-presence score back to Drive.

**What it does not do yet:** train anything. Training needs labelled text-line crops, which we don't have. This notebook produces the baseline and the data we'd need to label.

**Before you run:** upload `gst-sample-invoices.zip` to `My Drive/gst-engine/`.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
# PaddleOCR CPU build. Colab's Python version can change; if this cell fails, paste the error.
!pip install -q paddlepaddle==2.6.2 paddleocr==2.10.0 pymupdf


In [ ]:
import os, zipfile
from pathlib import Path

BASE = Path('/content/drive/MyDrive/gst-engine')
ZIP = BASE / 'gst-sample-invoices.zip'
WORK = Path('/content/gst-sample-invoices')
OUT = BASE / 'outputs'
OUT.mkdir(parents=True, exist_ok=True)

assert ZIP.exists(), f'Upload gst-sample-invoices.zip to {BASE} first'
if not WORK.exists():
    with zipfile.ZipFile(ZIP) as z:
        z.extractall('/content')
photos = sorted((WORK / 'internet_samples').glob('internet_gst_invoice_*.jpg'))
print('photos:', len(photos))


In [ ]:
from paddleocr import PaddleOCR
import json

ocr = PaddleOCR(lang='en', ocr_version='PP-OCRv5', use_angle_cls=True, show_log=False)
text_dir = OUT / 'paddle_text'
text_dir.mkdir(exist_ok=True)

for p in photos:
    result = ocr.ocr(str(p), cls=True)
    lines = [line[1][0] for page in (result or []) for line in (page or [])]
    (text_dir / (p.name + '.txt')).write_text('\n'.join(lines))
    print('ok', p.name, len(lines), 'lines')


In [ ]:
import re
gt = json.load(open(WORK / 'internet_samples' / 'internet_samples_ground_truth.json'))
gt = {s['filename']: s['ground_truth']['gt_parse']['header'] for s in gt}

def norm(s):
    return re.sub(r'[^A-Z0-9]', '', str(s or '').upper())

hits = {'gstin_no': [0, 0], 'invoice_date': [0, 0]}
for p in photos:
    text = norm((text_dir / (p.name + '.txt')).read_text())
    h = gt[p.name]
    for key in hits:
        if h.get(key):
            hits[key][1] += 1
            hits[key][0] += norm(h[key]) in text

for key, (ok, n) in hits.items():
    print(f'{key}: {ok}/{n} found in OCR text')
print('Caveat: ground-truth GSTIN labels are unreliable (some hold the buyer GSTIN). Read results with care.')
json.dump(hits, open(OUT / 'baseline_scores.json', 'w'), indent=2)


## Next step (needs your labels)

Fine-tuning a recognizer needs **text-line crops with the correct text**. To build them:
1. Export line crops from the OCR output above.
2. Correct the text by hand for about 200 real lines (GSTINs, dates, amounts).
3. Save them to `MyDrive/gst-engine/labels/`.

Once that exists, the training cell gets added here. Don't train on the earlier synthetic set, which is what caused the Donut failure.